# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL for this dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset (this fetches and parses the Croissant metadata)
dataset = mlc.Dataset(croissant_url)
# The metadata property provides a pydantic model - to view as dict, use .model_dump()
metadata = dataset.metadata.model_dump()
print(f"Loaded dataset: {metadata['name']}")
print(f"Description: {metadata['description'][:300]}...")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s.

Each record set in a Croissant dataset describes a table or logical collection of records, usually corresponding to a data file. Fields represent the structured columns. Let's inspect the available record sets and their fields using the Croissant metadata.

In [ ]:
# Print all top-level record sets and their fields, referencing all by @id

print("Record Sets in the dataset:")
record_sets = metadata.get('recordSet', [])
for record_set in record_sets:
    print(f"- Record Set @id: {record_set.get('@id')}, name: {record_set.get('name')}")
    fields = record_set.get('field', [])
    collist = []
    for field in fields:
        print(f"    Field @id: {field.get('@id')}, name: {field.get('name')}, data type: {field.get('dataType')}")
        collist.append(field.get('@id'))
    print(f"    Total fields: {len(fields)} | All field @id's: {collist}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# In the FAIR^2 dataset, get all record_set @id's for demonstration
record_sets = metadata.get('recordSet', [])
record_set_ids = [rs['@id'] for rs in record_sets]

# Load all as DataFrames using their @id
dataframes = {}
for record_set in record_set_ids:
    records = list(dataset.records(record_set=record_set))
    df = pd.DataFrame(records)
    dataframes[record_set] = df
    print(f"Loaded {len(df)} rows for record set @id: {record_set}")

# We'll use the first record set for demonstration
if record_set_ids:
    example_record_set = record_set_ids[0]
    print(f"Columns for {example_record_set}:")
    print(list(dataframes[example_record_set].columns))
    display(dataframes[example_record_set].head())
else:
    print("No record sets found in metadata.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.
**Note**: All columns/fields used are referenced by their `@id` as per Croissant specification.

In [ ]:
# EDA example using first available record set's numeric columns
import numpy as np

# Select a record set and its available fields by @id
if record_set_ids:
    rs_id = record_set_ids[0]
    df = dataframes[rs_id].copy()
    # Try to auto-detect a numeric field (float/int)
    # Strategy: from metadata, pick first numeric field
    chosen_numeric_field = None
    group_field = None
    record_set_md = None
    for rs in metadata['recordSet']:
        if rs['@id'] == rs_id:
            record_set_md = rs
            break
    if record_set_md:
        for field in record_set_md['field']:
            if field.get('dataType') in ['Float', 'Integer', 'Number']:
                chosen_numeric_field = field['@id']
                break
        # Also pick a string field as possible group_field
        for field in record_set_md['field']:
            if field.get('dataType') == 'Text':
                group_field = field['@id']
                break

    if chosen_numeric_field and (chosen_numeric_field in df.columns):
        threshold = df[chosen_numeric_field].mean()  # Example threshold: mean
        print(f"Filtering on field @id: {chosen_numeric_field} > {threshold:.2f}")
        filtered_df = df[df[chosen_numeric_field] > threshold].copy()
        print(f"Filtered rows: {len(filtered_df)}")
        # Normalize this field
        norm_col = f"{chosen_numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[chosen_numeric_field] - filtered_df[chosen_numeric_field].mean()) / filtered_df[chosen_numeric_field].std()
        print(filtered_df[[chosen_numeric_field, norm_col]].head())
        # Group by another field if available and numeric field is present
        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[chosen_numeric_field].mean().reset_index()
            print(f"Grouped mean {chosen_numeric_field} by {group_field}:")
            print(grouped_df.head())
    else:
        print("No numeric field found for EDA.")
else:
    print("No record sets available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using the extracted DataFrame.

We'll use matplotlib and seaborn for demonstration.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and chosen_numeric_field and (chosen_numeric_field in df.columns):
    plt.figure(figsize=(8,4))
    sns.histplot(df[chosen_numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {chosen_numeric_field}")
    plt.xlabel(chosen_numeric_field)
    plt.show()
    if group_field and group_field in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=chosen_numeric_field, data=df)
        plt.title(f"{chosen_numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Not enough data for visualization.")

## 6. Conclusion
This notebook demonstrated use of the `mlcroissant` library to:

- Load a Croissant dataset via its online schema.
- Enumerate record sets and fields by their `@id` for cross-referencing and data extraction.
- Load tabular data into Pandas DataFrames keyed by the Croissant record set `@id`.
- Apply basic exploratory data analysis: filtering, normalization, and grouping by field `@id`.
- Visualize the distribution of a numeric field and its relation to other fields.

You can extend this notebook for downstream analysis and machine learning tasks leveraging the dataset's rich structure and FAIR metadata.